# Reconstruction statistics and model complexity

R² measures reconstruction relative to each voxel's temporal mean; pooled R² weights subjects by their total squared variation. It can be negative. AIC, AICc and BIC trade residual fit against a nominal parameter count; smaller is better. Always compare identical data, masks and preprocessing.

For independent Gaussian residuals with one shared fitted variance, n scalar measurements and p parameters (including variance):

$$\hat\sigma^2=RSS/n,\quad \ell=-n[\log(2\pi)+1+\log(RSS/n)]/2.$$
$$AIC=2p-2\ell,\quad BIC=p\log n-2\ell,\quad AICc=AIC+2p(p+1)/(n-p-1).$$

AICc is infinite when its denominator is nonpositive. Perfect reconstruction has unbounded likelihood; constant observations have undefined R².

These formulas follow the [statsmodels implementation](https://www.statsmodels.org/v0.14.0/_modules/statsmodels/tools/eval_measures.html).

In [2]:
# Install htfa from PyPI if needed (for example on Colab).
try:
    import htfa  # noqa: F401
except ImportError:
    %pip install -q htfa

In [3]:
import numpy as np
import pandas as pd

import htfa
from htfa.stats import (
    gaussian_fit_statistics,
    parameter_count,
    reconstruction_statistics,
)

rng = np.random.default_rng(3)
x = rng.normal(size=(40, 30))
predicted = 0.7 * x
pd.DataFrame(
    [
        gaussian_fit_statistics(x, predicted, n_parameters=p).to_dict()
        for p in [5, 25, 100]
    ]
)

,n_observations,n_parameters,rss,tss,r2,log_likelihood,aic,aicc,bic
0,1200,5,107.043306,1158.613294,0.907611,-252.620226,515.240452,515.290703,540.690836
1,1200,25,107.043306,1158.613294,0.907611,-252.620226,555.240452,556.347777,682.492373
2,1200,100,107.043306,1158.613294,0.907611,-252.620226,705.240452,723.620798,1214.248136


The predictions are identical above: R² stays fixed while the information criteria penalize added parameters. This isolates the penalty from changes in fit.

## Counting a fitted model
For S subjects, K factors, M components and D coordinates, spatial parameters are
$S K[M(D+1)+(M-1)]$. The normalized mixing weights have M−1 free parameters per factor. Temporal weights add $K\sum_s T_s$; shared residual variance adds one. Component label permutations add no continuous parameters.

The helper counts **conditional reconstruction parameters**: it conditions on global/group templates and prior dispersions. It does not count those as independent reconstruction parameters or compute hierarchical marginal evidence. The count is nominal, not effective degrees of freedom under regularization.

In [4]:
xs, rs, truth = htfa.make_htfa_data(
    n_subjects=3, grid_shape=(12, 12), K=3, n_samples=40, random_state=0, n_components=2
)
model = htfa.HTFA(K=3, n_components=2, max_global_iter=3, random_state=0).fit(xs, rs)
result = htfa.HTFAResults(
    model, [htfa.Subject(x, r) for x, r in zip(xs, rs, strict=True)]
)
print(parameter_count(result))
pd.Series(reconstruction_statistics(result).to_dict())

{'spatial': 63, 'temporal': 360, 'variance': 1, 'total': 424}


/var/folders/tp/qtzc39jx5w556wl5w3dj21wr0000gn/T/ipykernel_68951/1652901506.py:4: ConvergenceWarning: HTFA did not converge in max_global_iter=3 iterations; increase max_global_iter or threshold
  model = htfa.HTFA(K=3, n_components=2, max_global_iter=3, random_state=0).fit(xs, rs)


n_observations    17280.000000
n_parameters        424.000000
rss               14369.460766
tss               17280.000000
r2                    0.168434
log_likelihood   -22925.656605
aic               46699.313210
aicc              46720.695589
bic               49988.410548
dtype: float64

## Interpretation for HTFA

HTFA fits are regularized, fMRI residuals are correlated, and overlapping mixtures can be singular. These plug-in criteria are heuristic conditional comparisons, not calibrated Bayes factors. n counts voxel-time entries, not independent subjects. Do not substitute a guessed effective n into just the penalty.

[Watanabe (2013)](https://jmlr.org/beta/papers/v14/watanabe13a.html) explains why ordinary BIC can fail for singular models. WAIC/LOO require a suitable posterior likelihood, which this point-estimate solver does not provide.

For stronger selection evidence use repeated starts and blocked held-out evaluation, with preprocessing fitted only on training data. Estimating test-time temporal weights measures spatial reconstruction, not forecasting. Tutorial 15 reports descriptive full-run scores for one dataset.